In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk import FreqDist
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from collections import Counter
from itertools import chain
import contractions
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder
from nltk.collocations import TrigramAssocMeasures, TrigramCollocationFinder
from nltk import ngrams
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("dataset/mb_data.csv")
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
_classes = data.type.unique()
print(_classes)

In [6]:
def show_class_distribution(
    data,
    x="type",
    figsize=(16, 4),
    title="Distribution of Personality Types",
    xticks_size=10,
    palette="husl",
):
    plt.figure(figsize=figsize)
    sns.countplot(x=x, data=data, palette=palette)
    plt.xlabel("Personality Types", size=15)
    plt.ylabel("Counts", size=15)
    plt.xticks(size=xticks_size)
    plt.title(title, size=20)
    plt.show()

In [ ]:
show_class_distribution(data, xticks_size=14)

In [8]:
def divide_types(df):
    df["E-I"] = ""
    df["N-S"] = ""
    df["F-T"] = ""
    df["J-P"] = ""
    for index, row in df.iterrows():
        row["E-I"] = "E" if row.type[0] == "E" else "I"
        row["N-S"] = "N" if row.type[1] == "N" else "S"
        row["F-T"] = "F" if row.type[2] == "F" else "T"
        row["J-P"] = "J" if row.type[3] == "J" else "P"
    return df


data = divide_types(data)

In [ ]:
def divide_types(df):
    df["E-I"] = df["type"].apply(lambda x: x[0])
    df["N-S"] = df["type"].apply(lambda x: x[1])
    df["F-T"] = df["type"].apply(lambda x: x[2])
    df["J-P"] = df["type"].apply(lambda x: x[3])
    return df


data = divide_types(data)
print(data["E-I"].value_counts())

In [ ]:
def show_class_distribution(
    data, x, title, figsize=(10, 6), xticks_size=15, palette="icefire"
):
    plt.figure(figsize=figsize)
    order = ["E", "I", "N", "S", "F", "T", "J", "P"]  # Force both categories
    sns.countplot(x=x, data=data, order=order, hue=x, palette=palette, legend=False)
    plt.title(title, fontsize=20)
    plt.xticks(fontsize=xticks_size)
    plt.xlabel(x, fontsize=15)
    plt.ylabel("Count", fontsize=15)
    plt.show()


show_class_distribution(
    data,
    x="E-I",
    title="Distribution of I & E",
    figsize=(9, 3),
    xticks_size=20,
    palette="icefire",
)

In [ ]:
show_class_distribution(
    data,
    x="N-S",
    title="Distribution of N & S",
    figsize=(9, 3),
    xticks_size=20,
    palette="cubehelix",
)

In [ ]:
show_class_distribution(
    data,
    x="F-T",
    title="Distribution of F & T",
    figsize=(9, 3),
    xticks_size=20,
    palette="viridis",
)

In [ ]:
show_class_distribution(
    data,
    x="J-P",
    title="Distribution of J & P",
    figsize=(9, 3),
    xticks_size=20,
    palette="flare",
)

In [ ]:
data.loc[7, "posts"]

#### Cleaning

In [15]:
def fix_contractions(df, column_name="posts", new_column="cleaned_post"):
    df[new_column] = df[column_name].apply(lambda x: contractions.fix(x))
    return df


data = fix_contractions(data)

In [16]:
def clean_data(df, column_name = "cleaned_post"):
    df[column_name] = df[column_name].apply(lambda x: x.lower())
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'@([a-zA-Z0-9_]{1,50})', '', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'#([a-zA-Z0-9_]{1,50})', '', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'http[s]?://\S+', '', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'[^A-Za-z]+', ' ', x))
    df[column_name] = df[column_name].apply(lambda x: re.sub(r' +', ' ', x))
    df[column_name] = df[column_name].apply(lambda x: " ".join([word for word in x.split() if not len(word) <3]))
    return df

data = clean_data(data)

In [ ]:
data.loc[7, "cleaned_post"]

In [ ]:
data["words_count"] = data["cleaned_post"].apply(lambda x: len(x.split()))
data.head(5)

In [19]:
def plot_counts(df, column, xlabel):
    fig = plt.figure()
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    df[column].plot.hist(bins=25)

In [ ]:
plot_counts(data, column="words_count", xlabel="Words Count")

In [ ]:
data["char_count"] = data["cleaned_post"].apply(lambda x: len(x))
data.head(5)

In [ ]:
plot_counts(data, column="char_count", xlabel="Character Count")

In [23]:
stopword_list = stopwords.words("english")

In [24]:
def get_most_frequent(data, stop_words, column="cleaned_post", top=25):
    df = data[column].apply(
        lambda x: " ".join([word for word in x.split() if not word in stop_words])
    )
    counter = Counter(" ".join(df).split())
    return counter.most_common(top)

In [ ]:
most_frequents = get_most_frequent(data, stopword_list)
most_frequents[:10]

In [26]:
def show_most_frequents(most_frequent_words, top=20):
    most_frequent_df = pd.DataFrame(most_frequent_words)
    plt.figure(figsize=(16, 4))
    my_cmap = plt.get_cmap("viridis")
    plt.bar(
        x=most_frequent_df.iloc[:top, 0],
        height=most_frequent_df.iloc[:top, 1],
        color="slateblue",
    )
    plt.xlabel("Words", size=17)
    plt.ylabel("Counts", size=17)
    plt.title("Most Frequent Words", size=20)
    plt.show()

In [ ]:
show_most_frequents(most_frequents)

In [28]:
def show_wordcloud(data, stopword_list, column="cleaned_post"):
    fig = plt.figure(figsize=(15, 5))
    wordcloud = WordCloud(
        background_color="black", min_font_size=5, stopwords=stopword_list
    ).generate(data[column].to_string())
    plt.axis("off")
    plt.imshow(wordcloud)
    plt.show()

In [ ]:
show_wordcloud(data, stopword_list)

In [30]:
def show_sub_wordclouds(data, type_column, column, size, fig_size=(20, 15)):
    classes = data[type_column].unique()
    fig, ax = plt.subplots(len(classes), figsize=fig_size)
    j = 0
    for _class in classes:
        temp = data[data[type_column] == _class]
        wordcloud = WordCloud(background_color="black").generate(
            temp[column].to_string()
        )
        plt.subplot(*size, j + 1)
        plt.title(_class, size=25)
        plt.imshow(wordcloud)
        plt.axis("off")
        j += 1

In [ ]:
show_sub_wordclouds(data, type_column="type", column="cleaned_post", size=(4, 4))

In [ ]:
show_sub_wordclouds(
    data, type_column="E-I", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [ ]:
show_sub_wordclouds(
    data, type_column="N-S", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [ ]:
show_sub_wordclouds(
    data, type_column="F-T", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [ ]:
show_sub_wordclouds(
    data, type_column="J-P", column="cleaned_post", size=(1, 2), fig_size=(16, 8)
)

In [36]:
def get_ngrams(data, n_gram, new_column, column="cleaned_post"):
    data["tokenized"] = data[column].apply(lambda x: x.split())
    data["sw_removal"] = data["tokenized"].apply(
        lambda x: [y for y in x if not y in stopword_list]
    )
    data[new_column] = data["sw_removal"].apply(lambda x: list(ngrams(x, n_gram)))
    data.drop(columns=["tokenized", "sw_removal"], inplace=True)
    return data

In [ ]:
data = get_ngrams(data, n_gram=2, new_column="bigrams")
data.head()

In [ ]:
data = get_ngrams(data, n_gram=3, new_column="trigrams")
data.head()

In [39]:
def most_common_ngram(data, column, top=20):
    temp = []
    for index, row in data.iterrows():
        temp += row[column]
    most_common = Counter(temp).most_common(top)
    return most_common

In [40]:
def plot_n_grams(ngrams, title, top=20):
    ngram_df = pd.DataFrame(ngrams)
    ngram_df.iloc[:, 0] = ngram_df.iloc[:, 0].astype(str)
    plt.figure(figsize=(7, 7))
    plt.barh(y=ngram_df.iloc[:top, 0], width=ngram_df.iloc[:top, 1])
    plt.xlabel("Counts", size=17)
    plt.ylabel("Pairs", size=17)
    plt.title(title, size=20)
    plt.show()

In [ ]:
bigrams_most_common = most_common_ngram(data, "bigrams")
bigrams_most_common

In [ ]:
plot_n_grams(bigrams_most_common, title="Most Frequent Bigrams")

In [ ]:
trigrams_most_common = most_common_ngram(data, "trigrams")
trigrams_most_common

In [ ]:
plot_n_grams(trigrams_most_common, title="Most Frequent Trigrams")

In [45]:
def remove_stopwords(data, stopword_list, column="cleaned_post"):
    data[column] = data[column].apply(
        word_tokenize
    ) 
    data[column] = data[column].apply(
        lambda x: [word for word in x if not word in stopword_list]
    )
    return data

In [46]:
def apply_lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(w) for w in text]

In [47]:
def lemmatize(data, stopword_list, column="cleaned_post"):
    data[column] = data[column].apply(apply_lemmatization)
    data[column] = data[column].apply(" ".join)
    return data

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
# data = remove_stopwords(data, stopword_list)

In [50]:
data = lemmatize(data, stopword_list)

In [ ]:
data.head()

In [ ]:
training_data = data[["cleaned_post", "E-I", "N-S", "F-T", "J-P"]].copy()
training_data.head(5)

In [53]:
def make_dummies(data, columns=["E-I", "N-S", "F-T", "J-P"]):
    for column in columns:
        temp_dummy = pd.get_dummies(data[column], prefix="type")
        data = data.join(temp_dummy)
    return data

In [54]:
def make_dummies(data, columns=["E-I", "N-S", "F-T", "J-P"]):
    for column in columns:
        temp_dummy = pd.get_dummies(
            data[column], prefix=f"type_{column}"
        ) 
        data = data.join(
            temp_dummy, lsuffix="_left", rsuffix="_right"
        )
    return data

In [ ]:
training_data = make_dummies(training_data)
training_data.head()

Handling Imbalanced Data

In [56]:
X = training_data[["cleaned_post"]]
y = training_data.drop(columns=["cleaned_post"])

In [ ]:
def show_distribution(
    data,
    x=["E-I", "N-S", "F-T", "J-P"],
    fig_size=(16, 4),
    xticks_size=10,
    palette="husl",
):
    fig, ax = plt.subplots(len(x), figsize=fig_size)
    j = 0
    for _x in x:
        plt.subplot(1, 4, j + 1)
        sns.countplot(x=_x, data=data, hue=_x, palette=palette, legend=False)
        plt.xticks(size=xticks_size)
        j += 1

show_distribution(data)

In [58]:
from imblearn.over_sampling import RandomOverSampler

In [59]:
y_cat = pd.Categorical(y)

In [60]:
oversample = RandomOverSampler()
X_over, y_over = oversample.fit_resample(X, y_cat)

In [ ]:
print(y.columns)

In [62]:
y_ei = y.get("type_E", None)
y_ns = y.get("type_N", None)
y_ft = y.get("type_F", None)
y_jp = y.get("type_J", None)

In [63]:
for col in y.columns:
    if "E" in col or "I" in col:
        y_ei = y[col]
    elif "N" in col or "S" in col:
        y_ns = y[col]
    elif "F" in col or "T" in col:
        y_ft = y[col]
    elif "J" in col or "P" in col:
        y_jp = y[col]

In [64]:
X_over_ei, y_over_ei = oversample.fit_resample(X, y_ei)
X_over_ns, y_over_ns = oversample.fit_resample(X, y_ns)
X_over_ft, y_over_ft = oversample.fit_resample(X, y_ft)
X_over_jp, y_over_jp = oversample.fit_resample(X, y_jp)

In [66]:
y_over_ei = y_over_ei.map({True: "E", False: "I"})
y_over_ns = y_over_ns.map({True: "N", False: "S"})
y_over_ft = y_over_ft.map({True: "F", False: "T"})
y_over_jp = y_over_jp.map({True: "J", False: "P"})

In [ ]:
show_class_distribution(data=X_over_ei, x=y_over_ei, figsize=(7, 3), title="E-I")

In [ ]:
def show_class_distribution(y, title, figsize=(7, 3)):
    plt.figure(figsize=figsize)
    sns.countplot(x=y, hue=y, palette="icefire", legend=False)
    plt.title(title, fontsize=20)
    plt.xlabel("Category", fontsize=15)
    plt.ylabel("Count", fontsize=15)
    plt.xticks(fontsize=12)
    plt.show()


# Now call the function for each dimension
show_class_distribution(y_over_ei, "Distribution of E-I (Oversampled)")
show_class_distribution(y_over_ns, "Distribution of N-S (Oversampled)")
show_class_distribution(y_over_ft, "Distribution of F-T (Oversampled)")
show_class_distribution(y_over_jp, "Distribution of J-P (Oversampled)")

Train test split for each classes

In [70]:
from sklearn.model_selection import train_test_split

In [71]:
X_train_ei, X_test_ei, y_train_ei, y_test_ei = train_test_split(
    X_over_ei, y_over_ei, test_size=0.3, random_state=42
)
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X_over_ns, y_over_ns, test_size=0.3, random_state=42
)
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(
    X_over_ft, y_over_ft, test_size=0.3, random_state=42
)
X_train_jp, X_test_jp, y_train_jp, y_test_jp = train_test_split(
    X_over_jp, y_over_jp, test_size=0.3, random_state=42
)

In [72]:
X_train_ei = X_train_ei["cleaned_post"]
X_train_ns = X_train_ns["cleaned_post"]
X_train_ft = X_train_ft["cleaned_post"]
X_train_jp = X_train_jp["cleaned_post"]

In [73]:
X_test_ei = X_test_ei["cleaned_post"]
X_test_ns = X_test_ns["cleaned_post"]
X_test_ft = X_test_ft["cleaned_post"]
X_test_jp = X_test_jp["cleaned_post"]

In [74]:
y_train_ei.name, y_test_ei.name = "E-I", "E-I"
y_train_ns.name, y_test_ns.name = "N-S", "N-S"
y_train_ft.name, y_test_ft.name = "F-T", "F-T"
y_train_jp.name, y_test_jp.name = "J-P", "J-P"

In [75]:
y_all_train = [y_train_ei, y_train_ns, y_train_ft, y_train_jp]
y_all_test = [y_test_ei, y_test_ns, y_test_ft, y_test_jp]

TF - IDF Vectorize

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [78]:
vectorizer = TfidfVectorizer(max_features=10000)

In [ ]:
vectorizer.fit(X_train_ei)

In [ ]:
X_train_ei = vectorizer.transform(X_train_ei)
X_test_ei = vectorizer.transform(X_test_ei)

X_train_ns = vectorizer.transform(X_train_ns)
X_test_ns = vectorizer.transform(X_test_ns)

X_train_ft = vectorizer.transform(X_train_ft)
X_test_ft = vectorizer.transform(X_test_ft)

X_train_jp = vectorizer.transform(X_train_jp)
X_test_jp = vectorizer.transform(X_test_jp)

In [ ]:
x_all_train = [X_train_ei, X_train_ns, X_train_ft, X_train_jp]
x_all_test = [X_test_ei, X_test_ns, X_test_ft, X_test_jp]

In [ ]:
tf_idf = pd.DataFrame(X_test_ei.toarray(), columns=vectorizer.get_feature_names_out())
tf_idf.head(10)

Model Creation & Model Trainning & Model Saving

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost
import pickle
from sklearn import metrics

In [ ]:
def create_models():
    nb_clf = MultinomialNB(alpha=0.01)
    svm_clf = SVC(C=1.0, kernel="linear", degree=3, gamma="auto")
    dt_clf = DecisionTreeClassifier(max_depth=7)
    rf_clf = RandomForestClassifier(n_estimators=750)
    xgb_clf = xgboost.XGBClassifier(use_label_encoder=False)
    return {
        "NaiveBayes": nb_clf,
        "SVM": svm_clf,
        "DecisionTree": dt_clf,
        "RandomForest": rf_clf,
        "Xgboost": xgb_clf,
    }

Model performance evalution with accuracy & f1 & roc-auc score

In [ ]:
_metrics = [
    "Accuracy",
    "Accuracy",
    "Accuracy",
    "Accuracy",
    "Precision",
    "Precision",
    "Precision",
    "Precision",
    "Recall",
    "Recall",
    "Recall",
    "Recall",
    "F1-Score",
    "F1-Score",
    "F1-Score",
    "F1-Score",
    "Roc-Auc Score",
    "Roc-Auc Score",
    "Roc-Auc Score",
    "Roc-Auc Score",
]
_types = [
    "E-I",
    "N-S",
    "F-T",
    "J-P",
    "E-I",
    "N-S",
    "F-T",
    "J-P",
    "E-I",
    "N-S",
    "F-T",
    "J-P",
    "E-I",
    "N-S",
    "F-T",
    "J-P",
    "E-I",
    "N-S",
    "F-T",
    "J-P",
]
_columns = ["NaiveBayes", "SVM", "DecisionTree", "RandomForest", "Xgboost"]

In [ ]:
evaluation_df = pd.DataFrame(columns=_columns, index=[_metrics, _types])
evaluation_df

In [ ]:
models = create_models()
models

In [ ]:
for model_item in models.items():
    for X_train, X_test, y_train, y_test in zip(
        x_all_train, x_all_test, y_all_train, y_all_test
    ):
        # Model creation and prediction
        model = model_item[1]
        print(f"{model} is training for {y_train.name}...")
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        # Performance evaluation metrics
        evaluation_df.loc["Accuracy", y_train.name][model_item[0]] = round(
            metrics.accuracy_score(y_test, pred), 3
        )
        evaluation_df.loc["Precision", y_train.name][model_item[0]] = round(
            metrics.precision_score(y_test, pred), 3
        )
        evaluation_df.loc["Recall", y_train.name][model_item[0]] = round(
            metrics.recall_score(y_test, pred), 3
        )
        evaluation_df.loc["F1-Score", y_train.name][model_item[0]] = round(
            metrics.f1_score(y_test, pred), 3
        )
        evaluation_df.loc["Roc-Auc Score", y_train.name][model_item[0]] = round(
            metrics.roc_auc_score(y_test, pred), 3
        )
        # Save model
        filename = f"saved-models/{model_item[0]}_{y_test.name}.sav"
        pickle.dump(model, open(filename, "wb"))

In [ ]:
evaluation_df

In [ ]:
### Save Tf-Idf Vectorizer

In [ ]:
filename = "vectorizer.pkl"
pickle.dump(vectorizer, open(filename, "wb"))